In [ ]:
import numpy as np
import torch
from torch.nn import *
from glob import glob
from subprocess import Popen
class Add(Sequential):
    def forward(s, x):
        return sum(s(x)for s in s)
class F(Module):
    def __init__(s, f, *_):
        super().__init__()
        s.f = f
        s._ = _
    def forward(s, x):
        return s.f(x, *s._)
def triu_indices(n, k=0, d=2):
    if d==2:
        return torch.triu_indices(n,n,k)
    i = triu_indices(n-k,k,d-1)
    return torch.cat([i.repeat_interleave(n-k-i[-1],1),torch.cat([i[-1:,j:]for j in[0,*i[0].diff().nonzero()+1]],1)+1])
def tril_indices(n, k=0, d=2):
    if d==2:
        return torch.tril_indices(n,n,k)
    i = tril_indices(n+k,k,d-1)
    return torch.cat([i.repeat_interleave(i[-1]+1,1)+1,torch.cat([i[-1:,:j]for j in[*(i[1]==d-3).nonzero()[1:],i.size(1)]],1)])
A = [np.array(i,'I')for i in glob('deck/*.ydk')for i in[[i for i in open(i).read().split('\n!')[0].split('\n')if i.isdigit()]]if len(i)>=55]
try:
    I,m,o = torch.load('deck1.pk',weights_only=0)
except FileNotFoundError:
    I = np.sort(list(set(np.concat(A))))
    m = Sequential(
        Embedding(len(I), 64),
        F(torch.cumsum, 1),
        SiLU(True),
        Linear(64, len(I)),
        LogSoftmax(2)
    )
    o = torch.optim.Adam(m.parameters(),.001)
alias = np.array([i.split(b'|')for i in Popen('sqlite3;cards.cdb;select id,alias from datas'.split(';'),stdout=-1).stdout.read().split()],'I')
alias = alias[alias[:,0].searchsorted(I),1]
alias = np.where(I[_:=I.searchsorted(alias)]==alias,_,np.arange(len(I))).astype('h')
A = [torch.from_numpy(alias[I.searchsorted(A)])for A in A]

m = Sequential(
    Embedding(64*4,16),
    F(torch.cumsum, 1),
    SiLU(True),
    Linear(16,64*4)
),Sequential(
    Embedding(64*4,16),
    F(lambda x:functional.pad(functional.silu(x[:,(_:=tril_indices(x.size(1),-1,2)).T].sum(2),True).cumsum(1)[:,torch.cat([(_[1]==0).nonzero()[1:,0],torch.tensor(_.shape[1:2])])-1],(0,0,1,0))),
    Linear(16,16),
    SiLU(True),
    Linear(16,64*4)
),Sequential(
    Embedding(64*4,16),
    F(lambda x:functional.pad(functional.silu(x[:,(_:=tril_indices(x.size(1),-1,3)).T].sum(2),True).cumsum(1)[:,torch.cat([(_[1]==1).nonzero()[1:,0],torch.tensor(_.shape[1:2])])-1],(0,0,2,0))),
    Linear(16,16),
    SiLU(True),
    Linear(16,64*4)
)

In [ ]:
while True:
    L = 0
    for _ in range(1000):
        o.zero_grad()
        a = [A[i][torch.randperm(len(A[i]))].int()for i in torch.randint(len(A),[4])]
        x = torch.stack([i[:min(map(len,a))-1]for i in a])
        y = m(x)
        l = sum(y[a[i:].unique()].logsumexp(0)for a,y in zip(a,y)for i,y in enumerate(y,1))/-x.numel()
        l.backward()
        o.step()
        L += l.data
    torch.save((I,m,o),'deck1.pk')
    print(L/1000)